# Step 3: EEG Channel Inspection and Harmonization

This notebook inspects channel names and documents how they relate to the requested common 19-channel 10-20 configuration. It does not filter, resample, normalize, artifact-remove, segment, combine, or modify EEG recordings. It only reads recording headers and sidecar channel metadata, creates mapping reports, and saves derived CSV files.

The requested target channels are: `Fp1`, `Fp2`, `F7`, `F3`, `Fz`, `F4`, `F8`, `T3`, `C3`, `Cz`, `C4`, `T4`, `T5`, `P3`, `Pz`, `P4`, `T6`, `O1`, `O2`.

In [27]:
from pathlib import Path
import json
import re

import mne
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 180)

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "NOTEBOOK":
    NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "NOTEBOOK" else NOTEBOOK_DIR
HARMONIZED_PATH = PROJECT_ROOT / "processed_data" / "harmonized_metadata.csv"
harmonized_metadata = pd.read_csv(HARMONIZED_PATH, dtype=str)

TARGET_CHANNELS = [
    "Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8",
    "T3", "C3", "Cz", "C4", "T4", "T5", "P3",
    "Pz", "P4", "T6", "O1", "O2",
]
TARGET_KEYS = {name.lower(): name for name in TARGET_CHANNELS}
ALIAS_TO_TARGET = {"t7": "T3", "t8": "T4", "p7": "T5", "p8": "T6"}

print("MNE version:", mne.__version__)
print("Harmonized metadata rows:", len(harmonized_metadata))
print("Target channels:", TARGET_CHANNELS)

MNE version: 1.6.1
Harmonized metadata rows: 310
Target channels: ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2']


## 1. Read channel information without changing EEG data

MNE reads supported EEGLAB headers with `preload=False`, so signal arrays are not loaded. AD-Auditory MATLAB v7.3 `.set` files are handled through their BIDS `channels.tsv` and EEG JSON sidecars when MNE reports that the file format is unsupported. This fallback still inspects metadata only and does not alter the `.set` file.

In [28]:
def sidecar_paths(path):
    stem = path.name.replace("_eeg.set", "")
    return path.with_name(f"{stem}_channels.tsv"), path.with_name(f"{stem}_eeg.json")

def classify_channel(name):
    key = re.sub(r"[^a-z0-9]+", "", str(name).lower())
    non_eeg_tokens = ("eog", "ecg", "ekg", "emg", "stim", "event", "trigger", "trig", "status", "aux", "acc", "gsr", "resp")
    return "non_eeg" if any(token in key for token in non_eeg_tokens) else "eeg_candidate"

def read_channel_header(path):
    try:
        raw = mne.io.read_raw_eeglab(path, preload=False, verbose="ERROR")
        try:
            names = list(raw.ch_names)
            mne.create_info(ch_names=names, sfreq=float(raw.info["sfreq"]), ch_types=["eeg"] * len(names))
            return names, "mne_header", None
        finally:
            raw.close()
    except NotImplementedError:
        channels_path, json_path = sidecar_paths(path)
        if not channels_path.exists():
            raise
        channels_table = pd.read_csv(channels_path, sep="\t")
        names = channels_table["name"].dropna().astype(str).tolist()
        if json_path.exists():
            with json_path.open(encoding="utf-8") as handle:
                json.load(handle)
        mne.create_info(ch_names=names, sfreq=1.0, ch_types=["eeg"] * len(names))
        return names, "bids_sidecar", "MNE does not read MATLAB v7.3 payload; channel metadata read from BIDS sidecars"

def standard_channel(source_name, dataset):
    if dataset == "BrainLat":
        return None, "extra_numeric_montage"
    key = str(source_name).strip().lower()
    if key in TARGET_KEYS:
        return TARGET_KEYS[key], "exact"
    if key in ALIAS_TO_TARGET:
        return ALIAS_TO_TARGET[key], "mapped"
    return None, "extra"

def montage_evidence(path):
    _, json_path = sidecar_paths(path)
    if json_path.exists():
        try:
            with json_path.open(encoding="utf-8") as handle:
                metadata = json.load(handle)
            return metadata.get("EEGPlacementScheme", metadata.get("CapManufacturer", "not specified"))
        except (OSError, json.JSONDecodeError):
            pass
    return "not specified in recording sidecar"

channel_cache = {}
channel_rows = []
for row in harmonized_metadata.itertuples(index=False):
    path = Path(row.eeg_file)
    cache_key = str(path)
    if cache_key not in channel_cache:
        try:
            names, source, note = read_channel_header(path)
            channel_cache[cache_key] = {"names": names, "source": source, "note": note, "error": None, "montage": montage_evidence(path)}
        except Exception as exc:
            channel_cache[cache_key] = {"names": [], "source": "failed", "note": None, "error": f"{type(exc).__name__}: {exc}", "montage": "unknown"}
    channel_rows.append({"dataset": row.dataset, "subject_id": row.subject_id, "eeg_file": row.eeg_file, **channel_cache[cache_key]})

channel_inventory = pd.DataFrame(channel_rows)
channel_inventory["original_channels"] = channel_inventory["names"].map(lambda values: "; ".join(values))
channel_inventory["channel_count"] = channel_inventory["names"].map(len)
channel_inventory["channel_types"] = channel_inventory["names"].map(lambda values: "; ".join(f"{name}:{classify_channel(name)}" for name in values))
display(channel_inventory[["dataset", "subject_id", "eeg_file", "channel_count", "original_channels", "source", "montage", "error"]].head(10))

,dataset,subject_id,eeg_file,channel_count,original_channels,source,montage,error
0,ADFTD,sub-001,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set,19,Fp1; Fp2; F3; F4; C3; C4; P3; P4; O1; O2; F7; F8; T3; T4; T5; T6; Fz; Cz; Pz,mne_header,not specified in recording sidecar,None
1,ADFTD,sub-001,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-001/eeg/sub-001_task-eyesclosed_eeg.set,19,Fp1; Fp2; F3; F4; C3; C4; P3; P4; O1; O2; F7; F8; T3; T4; T5; T6; Fz; Cz; Pz,mne_header,10-20,None
2,ADFTD,sub-002,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-002/eeg/sub-002_task-eyesclosed_eeg.set,19,Fp1; Fp2; F3; F4; C3; C4; P3; P4; O1; O2; F7; F8; T3; T4; T5; T6; Fz; Cz; Pz,mne_header,not specified in recording sidecar,None
3,ADFTD,sub-002,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-002/eeg/sub-002_task-eyesclosed_eeg.set,19,Fp1; Fp2; F3; F4; C3; C4; P3; P4; O1; O2; F7; F8; T3; T4; T5; T6; Fz; Cz; Pz,mne_header,10-20,None
4,ADFTD,sub-003,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set,19,Fp1; Fp2; F3; F4; C3; C4; P3; P4; O1; O2; F7; F8; T3; T4; T5; T6; Fz; Cz; Pz,mne_header,not specified in recording sidecar,None
5,ADFTD,sub-003,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-003/eeg/sub-003_task-eyesclosed_eeg.set,19,Fp1; Fp2; F3; F4; C3; C4; P3; P4; O1; O2; F7; F8; T3; T4; T5; T6; Fz; Cz; Pz,mne_header,10-20,None
6,ADFTD,sub-004,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-004/eeg/sub-004_task-eyesclosed_eeg.set,19,Fp1; Fp2; F3; F4; C3; C4; P3; P4; O1; O2; F7; F8; T3; T4; T5; T6; Fz; Cz; Pz,mne_header,not specified in recording sidecar,None
7,ADFTD,sub-004,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-004/eeg/sub-004_task-eyesclosed_eeg.set,19,Fp1; Fp2; F3; F4; C3; C4; P3; P4; O1; O2; F7; F8; T3; T4; T5; T6; Fz; Cz; Pz,mne_header,10-20,None
8,ADFTD,sub-005,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-005/eeg/sub-005_task-eyesclosed_eeg.set,19,Fp1; Fp2; F3; F4; C3; C4; P3; P4; O1; O2; F7; F8; T3; T4; T5; T6; Fz; Cz; Pz,mne_header,not specified in recording sidecar,None
9,ADFTD,sub-005,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-005/eeg/sub-005_task-eyesclosed_eeg.set,19,Fp1; Fp2; F3; F4; C3; C4; P3; P4; O1; O2; F7; F8; T3; T4; T5; T6; Fz; Cz; Pz,mne_header,10-20,None


## 2. Create explicit dataset channel mappings

Each source channel is classified as an exact target match, an explicit anatomical alias, or an extra channel. Missing target channels are represented separately in the mapping table. No missing electrode is synthesized and no source channel is duplicated.

In [29]:
channel_mapping = {}
mapping_rows = []
for dataset in sorted(channel_inventory["dataset"].unique()):
    names = sorted({name for values in channel_inventory.loc[channel_inventory["dataset"] == dataset, "names"] for name in values})
    channel_mapping[dataset] = {}
    found_targets = set()
    for name in names:
        standard, status = standard_channel(name, dataset)
        if standard is not None:
            found_targets.add(standard)
            channel_mapping[dataset][name] = standard
        mapping_rows.append({
            "Dataset": dataset,
            "Original Channel Name": name,
            "Standard Channel Name": standard if standard is not None else "",
            "Mapping Status": status,
        })
    for target in TARGET_CHANNELS:
        if target not in found_targets:
            mapping_rows.append({
                "Dataset": dataset,
                "Original Channel Name": "",
                "Standard Channel Name": target,
                "Mapping Status": "missing",
            })

channel_mapping_table = pd.DataFrame(mapping_rows)
print("Channel mapping dictionary:")
display(channel_mapping_table)

mapping_target_rows = channel_mapping_table[channel_mapping_table["Mapping Status"].isin(["exact", "mapped"])]
duplicate_target_details = mapping_target_rows.groupby(["Dataset", "Standard Channel Name"])["Original Channel Name"].nunique()
duplicate_target_details = duplicate_target_details[duplicate_target_details > 1]
print("Duplicate target channels by dataset:")
display(duplicate_target_details)
assert len(duplicate_target_details) == 0

Channel mapping dictionary:


,Dataset,Original Channel Name,Standard Channel Name,Mapping Status
0,ADFTD,C3,C3,exact
1,ADFTD,C4,C4,exact
2,ADFTD,Cz,Cz,exact
3,ADFTD,F3,F3,exact
4,ADFTD,F4,F4,exact
...,...,...,...,...
180,BrainLat,,Pz,missing
181,BrainLat,,P4,missing
182,BrainLat,,T6,missing
183,BrainLat,,O1,missing


Duplicate target channels by dataset:


Series([], Name: Original Channel Name, dtype: int64)

## 3. Report channel availability and harmonization options

Availability is calculated for every recording. Non-EEG channels are reported but excluded from target matching. The notebook does not automatically choose a common-channel strategy, full-montage restriction, or variable-channel model architecture.

In [30]:
def recording_availability(dataset, names):
    eeg_names = [name for name in names if classify_channel(name) == "eeg_candidate"]
    mapped = {}
    for name in eeg_names:
        standard, _ = standard_channel(name, dataset)
        if standard is not None:
            mapped[standard] = name
    found = [target for target in TARGET_CHANNELS if target in mapped]
    missing = [target for target in TARGET_CHANNELS if target not in mapped]
    extras = [name for name in eeg_names if standard_channel(name, dataset)[0] is None]
    return eeg_names, found, missing, extras

availability_rows = []
for row in channel_inventory.itertuples(index=False):
    eeg_names, found, missing, extras = recording_availability(row.dataset, row.names)
    availability_rows.append({
        "Dataset": row.dataset,
        "Subject_ID": row.subject_id,
        "EEG_File": row.eeg_file,
        "Original_Channels": row.channel_count,
        "EEG_Channels": len(eeg_names),
        "Target_Channels_Found": "; ".join(found),
        "Missing_Channels": "; ".join(missing),
        "Extra_Channels": "; ".join(extras),
        "Target_Availability_Percent": round(100 * len(found) / len(TARGET_CHANNELS), 2),
        "Full_Target_Montage": len(found) == len(TARGET_CHANNELS),
        "Header_Source": row.source,
        "Read_Error": row.error,
    })
availability_report = pd.DataFrame(availability_rows)

dataset_channel_report = (
    availability_report.groupby("Dataset")
    .agg(
        Original_Channels=("Original_Channels", lambda values: ", ".join(sorted({str(value) for value in values}))),
        EEG_Channels=("EEG_Channels", lambda values: ", ".join(sorted({str(value) for value in values}))),
        Target_Channels_Found=("Target_Channels_Found", lambda values: ", ".join(sorted(set(values)))),
        Missing_Channels=("Missing_Channels", lambda values: ", ".join(sorted(set(values)))),
        Extra_Channels=("Extra_Channels", lambda values: ", ".join(sorted(set(values)))),
    )
    .reset_index()
)
print("Dataset channel availability report:")
display(dataset_channel_report)
print("Per-recording channel availability:")
display(availability_report.head(20))

dataset_montage_report = channel_inventory.groupby("dataset")["montage"].agg(lambda values: ", ".join(sorted(set(values))).strip())
print("Montage evidence by dataset:")
display(dataset_montage_report.rename("Montage evidence").reset_index())

print("Decision options for the next step; none is selected automatically:")
print("A. Use only the common channels available across all datasets.")
print("B. Restrict the final model to recordings with the full target montage.")
print("C. Use a model architecture capable of handling variable channel sets.")

Dataset channel availability report:


,Dataset,Original_Channels,EEG_Channels,Target_Channels_Found,Missing_Channels,Extra_Channels
0,ADFTD,19,19,Fp1; Fp2; F7; F3; Fz; F4; F8; T3; C3; Cz; C4; T4; T5; P3; Pz; P4; T6; O1; O2,,
1,AD_Auditory,19,19,Fp1; Fp2; F7; F3; Fz; F4; F8; T3; C3; Cz; C4; T4; T5; P3; Pz; P4; T6; O1; O2,,
2,BrainLat,128,128,,Fp1; Fp2; F7; F3; Fz; F4; F8; T3; C3; Cz; C4; T4; T5; P3; Pz; P4; T6; O1; O2,A1; A2; A3; A4; A5; A6; A7; A8; A9; A10; A11; A12; A13; A14; A15; A16; A17; A18; A19; A20; A21; A22; A23; A24; A25; A26; A27; A28; A29; A30; A31; A32; B1; B2; B3; B4; B5; B6; B...


Per-recording channel availability:


,Dataset,Subject_ID,EEG_File,Original_Channels,EEG_Channels,Target_Channels_Found,Missing_Channels,Extra_Channels,Target_Availability_Percent,Full_Target_Montage,Header_Source,Read_Error
0,ADFTD,sub-001,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set,19,19,Fp1; Fp2; F7; F3; Fz; F4; F8; T3; C3; Cz; C4; T4; T5; P3; Pz; P4; T6; O1; O2,,,100.0,True,mne_header,None
1,ADFTD,sub-001,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-001/eeg/sub-001_task-eyesclosed_eeg.set,19,19,Fp1; Fp2; F7; F3; Fz; F4; F8; T3; C3; Cz; C4; T4; T5; P3; Pz; P4; T6; O1; O2,,,100.0,True,mne_header,None
2,ADFTD,sub-002,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-002/eeg/sub-002_task-eyesclosed_eeg.set,19,19,Fp1; Fp2; F7; F3; Fz; F4; F8; T3; C3; Cz; C4; T4; T5; P3; Pz; P4; T6; O1; O2,,,100.0,True,mne_header,None
3,ADFTD,sub-002,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-002/eeg/sub-002_task-eyesclosed_eeg.set,19,19,Fp1; Fp2; F7; F3; Fz; F4; F8; T3; C3; Cz; C4; T4; T5; P3; Pz; P4; T6; O1; O2,,,100.0,True,mne_header,None
4,ADFTD,sub-003,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set,19,19,Fp1; Fp2; F7; F3; Fz; F4; F8; T3; C3; Cz; C4; T4; T5; P3; Pz; P4; T6; O1; O2,,,100.0,True,mne_header,None
5,ADFTD,sub-003,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-003/eeg/sub-003_task-eyesclosed_eeg.set,19,19,Fp1; Fp2; F7; F3; Fz; F4; F8; T3; C3; Cz; C4; T4; T5; P3; Pz; P4; T6; O1; O2,,,100.0,True,mne_header,None
6,ADFTD,sub-004,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-004/eeg/sub-004_task-eyesclosed_eeg.set,19,19,Fp1; Fp2; F7; F3; Fz; F4; F8; T3; C3; Cz; C4; T4; T5; P3; Pz; P4; T6; O1; O2,,,100.0,True,mne_header,None
7,ADFTD,sub-004,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-004/eeg/sub-004_task-eyesclosed_eeg.set,19,19,Fp1; Fp2; F7; F3; Fz; F4; F8; T3; C3; Cz; C4; T4; T5; P3; Pz; P4; T6; O1; O2,,,100.0,True,mne_header,None
8,ADFTD,sub-005,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-005/eeg/sub-005_task-eyesclosed_eeg.set,19,19,Fp1; Fp2; F7; F3; Fz; F4; F8; T3; C3; Cz; C4; T4; T5; P3; Pz; P4; T6; O1; O2,,,100.0,True,mne_header,None
9,ADFTD,sub-005,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-005/eeg/sub-005_task-eyesclosed_eeg.set,19,19,Fp1; Fp2; F7; F3; Fz; F4; F8; T3; C3; Cz; C4; T4; T5; P3; Pz; P4; T6; O1; O2,,,100.0,True,mne_header,None


Montage evidence by dataset:


,dataset,Montage evidence
0,ADFTD,"10-20, not specified in recording sidecar"
1,AD_Auditory,10-20
2,BrainLat,not specified in recording sidecar


Decision options for the next step; none is selected automatically:
A. Use only the common channels available across all datasets.
B. Restrict the final model to recordings with the full target montage.
C. Use a model architecture capable of handling variable channel sets.


## 4. Save reports

The CSV files are derived reports only. The original EEG recordings and raw metadata remain unchanged.

In [31]:
output_dir = PROJECT_ROOT / "processed_data"
output_dir.mkdir(parents=True, exist_ok=True)
mapping_output = output_dir / "channel_mapping.csv"
availability_output = output_dir / "channel_availability_report.csv"
channel_mapping_table.to_csv(mapping_output, index=False)
availability_report.to_csv(availability_output, index=False)
print(f"Saved channel mapping: {mapping_output}")
print(f"Saved channel availability report: {availability_output}")
full_montage_count = int(availability_report["Full_Target_Montage"].sum())
print(f"Recordings with full target montage: {full_montage_count} / {len(availability_report)}")
print("CHANNEL INSPECTION AND HARMONIZATION COMPLETE — READY FOR SAMPLING-RATE HARMONIZATION")

Saved channel mapping: /home/admin1/Desktop/SIH_2026_ALL/processed_data/channel_mapping.csv
Saved channel availability report: /home/admin1/Desktop/SIH_2026_ALL/processed_data/channel_availability_report.csv
Recordings with full target montage: 211 / 310
CHANNEL INSPECTION AND HARMONIZATION COMPLETE — READY FOR SAMPLING-RATE HARMONIZATION
